# Phase 5 — NB4: Diagnostic — Why Retrieval Loses to No-Retrieval

**Goal:** Identify root cause(s) of the 9.5pp gap (Sent Acc|CC: 0.8058 ret vs 0.9011 no-ret).

**7 experiments, priority-ordered:**
1. Exp 2: Retrieval quality (pol_match@K) on test vs train
2. Exp 1: Oracle category (gold cats → isolate Stage 1 errors)
3. Exp 4: Per-polarity breakdown + confusion matrices
4. Exp 3: Ablate text signal vs vector signal
5. Exp 5: Error analysis (hurt cases)
6. Exp 6: Confidence analysis
7. Exp 7: CLS 768-dim from no-ret DeBERTa as retrieval vector (test generalization hypothesis)

**Input:** `p5-nb1-stage1`, `p5-nb2-stage2`, `p5-embed-v4`

**Output:** Diagnostic report identifying primary root cause.

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml iterative-stratification

In [ ]:
import os, sys, json, shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
import torch, gc, math
import numpy as np
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 0b. Wire Checkpoints & Data

In [ ]:
def find_input(name):
    for p in [f'/kaggle/input/{name}', f'/kaggle/input/datasets/lcminhc/{name}']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'Dataset {name} not found')

NB1 = find_input('p5-nb1-stage1')
NB2 = find_input('p5-nb2-stage2')
EMB = find_input('p5-embed-v4')

print(f'NB1: {NB1} -> {os.listdir(NB1)}')
print(f'NB2: {NB2} -> {os.listdir(NB2)}')
print(f'EMB: {EMB} -> {os.listdir(EMB)}')

# Stage 1
STAGE1_CONFIG = 'configs/stage1_2014_cataware.yaml'
STAGE1_CKPT_NAME = 'stage1_2014_cataware_best.pt'
os.makedirs('checkpoints/stage1', exist_ok=True)
shutil.copy(f'{NB1}/{STAGE1_CKPT_NAME}', 'checkpoints/stage1/best.pt')

# Stage 2
os.makedirs('checkpoints/stage2_2014', exist_ok=True)
os.makedirs('checkpoints/stage2_2014_noret', exist_ok=True)
shutil.copy(f'{NB2}/stage2_2014_best.pt', 'checkpoints/stage2_2014/best.pt')
shutil.copy(f'{NB2}/stage2_2014_noret_best.pt', 'checkpoints/stage2_2014_noret/best.pt')

# Embedding
os.makedirs('checkpoints/embedding_2014', exist_ok=True)
shutil.copy(f'{EMB}/embedding_v4_s2_best.pt', 'checkpoints/embedding_2014/best.pt')

# Data
os.makedirs('data/processed', exist_ok=True)
shutil.copy(f'{NB1}/category_detection.jsonl', 'data/processed/category_detection.jsonl')
shutil.copy(f'{NB1}/sentiment_records.jsonl', 'data/processed/sentiment_records.jsonl')

# Build FAISS index (all 3516 train records — same as NB3 test-time)
os.makedirs('indexes', exist_ok=True)
!python scripts/03_build_index.py \
    --embedding_ckpt checkpoints/embedding_2014/best.pt \
    --input data/processed/sentiment_records.jsonl \
    --out_dir indexes/

print('\nAll files wired.')

## 0c. Load Data & Configs

In [ ]:
from src.utils.io import load_yaml, read_jsonl
from src.utils.seed import set_seed
from src.data.category_builder import CATEGORY_LIST, CAT2IDX, POL2ID

ID2POL = {v: k for k, v in POL2ID.items()}
set_seed(42)

s1_cfg = load_yaml(STAGE1_CONFIG)
s2_cfg = load_yaml('configs/stage2_2014.yaml')
s2_noret_cfg = load_yaml('configs/stage2_2014_noret.yaml')
ret_cfg = load_yaml('configs/retrieval_v2.yaml')

# Load all records
cat_records = read_jsonl(s1_cfg['category_path'])
sent_records = read_jsonl(s2_cfg['sentiment_path'])

test_cat = [r for r in cat_records if r['split'] == 'test']
test_sent = [r for r in sent_records if r['split'] == 'test']
train_sent = [r for r in sent_records if r['split'] == 'train']

print(f'Test cat records: {len(test_cat)}')
print(f'Test sent records: {len(test_sent)}')
print(f'Train sent records: {len(train_sent)}')

# Reconstruct Stage 2 train/val split (matches 04b_train_stage2.py)
unique_sents = list(set(r['sentence'] for r in train_sent))
train_sents_s2, val_sents_s2 = train_test_split(
    unique_sents, test_size=s2_cfg['val_ratio'], random_state=s2_cfg['seed'])
val_sents_set = set(val_sents_s2)
train_recs_s2 = [r for r in train_sent if r['sentence'] not in val_sents_set]
val_recs_s2 = [r for r in train_sent if r['sentence'] in val_sents_set]
print(f'Stage 2 train recs: {len(train_recs_s2)}, val recs: {len(val_recs_s2)}')

# Test polarity distribution
test_pol_dist = Counter(r['polarity'] for r in test_sent)
train_pol_dist = Counter(r['polarity'] for r in train_sent)
print(f'\nTrain polarity: {dict(train_pol_dist)}')
print(f'Test polarity:  {dict(test_pol_dist)}')

## 1. Load Embedding Model + FAISS Index

In [ ]:
from src.embedding.model import ContrastiveEmbedder
from src.retrieval.index import load_index, build_index
from src.retrieval.retriever import Retriever
from src.retrieval.encoder import encode_records
from transformers import AutoTokenizer

# Load embedding model (keep loaded throughout — needed for all retrieval experiments)
embed_model = ContrastiveEmbedder(model_name=s2_cfg['model_name'], proj_dim=256)
embed_model.load_state_dict(
    torch.load('checkpoints/embedding_2014/best.pt', map_location='cpu'), strict=False)
embed_model.to(DEVICE)
embed_model.eval()
print('Embedding model loaded')

# Load full FAISS index (3516 train records — test-time index)
full_index, full_metadata, full_vectors = load_index('indexes')
print(f'Full FAISS index: {full_index.ntotal} vectors')

# Build train-only index (2800 records — matches training conditions)
tokenizer = AutoTokenizer.from_pretrained(s2_cfg['model_name'])
train_only_vectors = encode_records(
    train_recs_s2, embed_model, tokenizer,
    max_length=128, batch_size=64, device=DEVICE)
train_only_index = build_index(train_only_vectors)
train_only_metadata = [{
    'id': r['id'], 'sentence': r['sentence'],
    'aspect_category': r.get('aspect_category', r.get('category')),
    'polarity': r['polarity']
} for r in train_recs_s2]
print(f'Train-only FAISS index: {train_only_index.ntotal} vectors')

---
## Exp 2: Retrieval Quality — Test vs Train

Measure pol_match@K, cat_match@K, avg cosine similarity for:
- Test records querying the full index (test-time conditions)
- Train records querying the train-only index with self-exclusion (training conditions)

In [ ]:
def measure_retrieval_quality(records, index, metadata, vectors, embed_model,
                              tokenizer, top_k=5, threshold=0.3,
                              use_exclude=True, device='cuda'):
    """Measure retrieval quality for a set of records."""
    retriever = Retriever(index, metadata, top_k=top_k, threshold=threshold)
    results = []
    
    for i, rec in enumerate(records):
        sentence = rec['sentence']
        category = rec.get('aspect_category', rec.get('category'))
        polarity = rec['polarity']
        
        tok_enc = tokenizer(
            sentence, category,
            max_length=128, padding=False, truncation=True,
            return_tensors='pt')
        with torch.no_grad():
            query_vec = embed_model.encode(
                tok_enc['input_ids'].to(device),
                tok_enc['attention_mask'].to(device))
        query_np = query_vec.cpu().numpy().astype('float32')
        
        exclude = sentence if use_exclude else None
        neighbors = retriever.retrieve(query_np, exclude_sentence=exclude)
        
        n_valid = len(neighbors)
        if n_valid == 0:
            results.append({
                'polarity': polarity, 'category': category,
                'n_valid': 0,
                'pol_match_1': 0, 'pol_match_3': 0, 'pol_match_5': 0,
                'cat_match_1': 0, 'cat_match_3': 0, 'cat_match_5': 0,
                'avg_sim': 0, 'neighbors': []
            })
            continue
        
        pol_matches = [1 if nb['polarity'] == polarity else 0 for nb in neighbors]
        cat_matches = [1 if nb.get('aspect_category', nb.get('category')) == category else 0
                       for nb in neighbors]
        scores = [nb['score'] for nb in neighbors]
        
        def match_at_k(matches, k):
            subset = matches[:k]
            return sum(subset) / len(subset) if subset else 0
        
        results.append({
            'polarity': polarity, 'category': category,
            'n_valid': n_valid,
            'pol_match_1': match_at_k(pol_matches, 1),
            'pol_match_3': match_at_k(pol_matches, 3),
            'pol_match_5': match_at_k(pol_matches, 5),
            'cat_match_1': match_at_k(cat_matches, 1),
            'cat_match_3': match_at_k(cat_matches, 3),
            'cat_match_5': match_at_k(cat_matches, 5),
            'avg_sim': np.mean(scores),
            'neighbors': [{'sentence': nb['sentence'][:60],
                           'category': nb.get('aspect_category', nb.get('category')),
                           'polarity': nb['polarity'],
                           'score': round(nb['score'], 4)}
                          for nb in neighbors]
        })
        if (i + 1) % 200 == 0:
            print(f'  Processed {i+1}/{len(records)}')
    
    return results


def print_quality_report(results, label):
    n = len(results)
    print(f'\n=== {label} ({n} records) ===')
    
    for metric in ['pol_match_1', 'pol_match_3', 'pol_match_5',
                   'cat_match_1', 'cat_match_3', 'cat_match_5', 'avg_sim']:
        vals = [r[metric] for r in results]
        print(f'  {metric:15s}: {np.mean(vals):.4f} (std={np.std(vals):.4f})')
    
    n_few = sum(1 for r in results if r['n_valid'] < 3)
    n_zero = sum(1 for r in results if r['n_valid'] == 0)
    print(f'  < 3 neighbors:   {n_few}/{n} ({100*n_few/n:.1f}%)')
    print(f'  0 neighbors:     {n_zero}/{n} ({100*n_zero/n:.1f}%)')
    
    # Breakdown by polarity
    print(f'\n  By polarity:')
    for pol in ['positive', 'negative', 'neutral']:
        subset = [r for r in results if r['polarity'] == pol]
        if not subset:
            continue
        pm5 = np.mean([r['pol_match_5'] for r in subset])
        sim = np.mean([r['avg_sim'] for r in subset])
        print(f'    {pol:10s} (n={len(subset):4d}): pol_match@5={pm5:.4f}, avg_sim={sim:.4f}')
    
    # Breakdown by category
    print(f'\n  By category:')
    for cat in CATEGORY_LIST:
        subset = [r for r in results if r['category'] == cat]
        if not subset:
            continue
        pm5 = np.mean([r['pol_match_5'] for r in subset])
        cm5 = np.mean([r['cat_match_5'] for r in subset])
        print(f'    {cat:30s} (n={len(subset):3d}): pol@5={pm5:.4f}, cat@5={cm5:.4f}')

In [ ]:
print('Measuring retrieval quality on TEST records (full index, no self-exclusion needed)...')
test_quality = measure_retrieval_quality(
    test_sent, full_index, full_metadata, full_vectors,
    embed_model, tokenizer, top_k=5, threshold=0.3,
    use_exclude=False, device=DEVICE)
print_quality_report(test_quality, 'TEST → Full Index')

In [ ]:
# Sample 500 train records for comparison (full set takes too long)
np.random.seed(42)
sample_idx = np.random.choice(len(train_recs_s2), min(500, len(train_recs_s2)), replace=False)
train_sample = [train_recs_s2[i] for i in sample_idx]

print('Measuring retrieval quality on TRAIN records (train-only index, with self-exclusion)...')
train_quality = measure_retrieval_quality(
    train_sample, train_only_index, train_only_metadata, train_only_vectors,
    embed_model, tokenizer, top_k=5, threshold=0.3,
    use_exclude=True, device=DEVICE)
print_quality_report(train_quality, 'TRAIN → Train-Only Index')

In [ ]:
# Side-by-side comparison
print('\n' + '=' * 60)
print('EXP 2 SUMMARY: Retrieval Quality Gap')
print('=' * 60)
for metric in ['pol_match_1', 'pol_match_3', 'pol_match_5', 'avg_sim']:
    train_val = np.mean([r[metric] for r in train_quality])
    test_val = np.mean([r[metric] for r in test_quality])
    gap = test_val - train_val
    print(f'  {metric:15s}  Train={train_val:.4f}  Test={test_val:.4f}  Gap={gap:+.4f}')
print()

---
## Exp 1 + 4: Oracle Category + Per-Polarity Breakdown

Compare:
- Retrieval with predicted categories (standard NB3)
- Retrieval with gold categories (oracle — isolates Stage 1 error)
- No-retrieval with predicted categories (baseline)

In [ ]:
from src.absa.category_model import CategoryDetector
from src.absa.category_dataset import CategoryDataset
from src.absa.sentiment_dataset import SentimentDataset
from src.absa.sentiment_model import SentimentPredictor
from src.absa.category_trainer import _tune_global_threshold, _apply_global_threshold
from src.evaluation.category_metrics import (
    category_f1, joint_category_sentiment_f1,
    sentiment_acc_given_correct_category,
)


def predict_sentiment_with_details(model, records, retriever, embedding_model,
                                   tokenizer_name, max_length, top_k, device,
                                   use_retrieval, store_vectors=None, batch_size=16):
    """Like predict_sentiment but also returns softmax probabilities."""
    from torch.utils.data import DataLoader
    embed_device = 'cpu'
    if embedding_model is not None:
        embed_device = next(embedding_model.parameters()).device.type
    ds = SentimentDataset(
        records, retriever=retriever,
        tokenizer_name=tokenizer_name,
        embedding_model=embedding_model,
        store_vectors=store_vectors,
        max_length=max_length,
        top_k=top_k, device=embed_device,
        use_retrieval=use_retrieval,
    )
    loader = DataLoader(ds, batch_size=batch_size)
    all_preds = []
    all_probs = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch_gpu = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                         for k, v in batch.items()}
            out = model(
                input_ids=batch_gpu['input_ids'],
                attention_mask=batch_gpu['attention_mask'],
                neighbor_polarities=batch_gpu.get('neighbor_polarities'),
                neighbor_scores=batch_gpu.get('neighbor_scores'),
                query_vec=batch_gpu.get('query_vec'),
                neighbor_vecs=batch_gpu.get('neighbor_vecs'),
                query_polarity=batch_gpu.get('query_polarity'),
            )
            logits = out['logits']
            probs = torch.softmax(logits, dim=-1).cpu()
            preds = logits.argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_probs.append(probs)
    all_probs = torch.cat(all_probs, dim=0)
    return all_preds, all_probs

In [ ]:
# --- Stage 1: predict categories on test ---
s1_model = CategoryDetector(
    model_name=s1_cfg['model_name'],
    num_categories=s1_cfg['num_categories'],
    use_cat_attention=s1_cfg.get('use_cat_attention', False),
).to(DEVICE)
s1_ckpt = torch.load('checkpoints/stage1/best.pt', map_location=DEVICE)
s1_model.load_state_dict(s1_ckpt['model_state'], strict=False)
s1_model.eval()

# Reconstruct val split for threshold tuning
train_cat = [r for r in cat_records if r['split'] == 'train']
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    label_matrix = np.array([r['category_vector'] for r in train_cat])
    msss = MultilabelStratifiedShuffleSplit(
        n_splits=1, test_size=s1_cfg['val_ratio'],
        random_state=s1_cfg['seed'])
    for _, val_idx in msss.split(label_matrix, label_matrix):
        val_cat = [train_cat[i] for i in val_idx]
except ImportError:
    stratify_key = [min(sum(r['category_vector']), 2) for r in train_cat]
    _, val_cat = train_test_split(
        train_cat, test_size=s1_cfg['val_ratio'],
        random_state=s1_cfg['seed'], stratify=stratify_key)

from torch.utils.data import DataLoader

def collect_logits(model, dataset, device, batch_size=32):
    loader = DataLoader(dataset, batch_size=batch_size)
    all_logits = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                     for k, v in batch.items()}
            out = model(batch['input_ids'], batch['attention_mask'])
            all_logits.append(out['logits'].cpu())
    return torch.cat(all_logits, dim=0)

val_ds = CategoryDataset(val_cat, tokenizer_name=s1_cfg['model_name'],
                         max_length=s1_cfg['max_seq_length'])
val_logits = collect_logits(s1_model, val_ds, DEVICE)
val_labels = torch.stack([torch.tensor(r['category_vector'], dtype=torch.float32)
                          for r in val_cat])
threshold = _tune_global_threshold(val_logits, val_labels)
print(f'Tuned global threshold: {threshold:.2f}')

test_ds = CategoryDataset(test_cat, tokenizer_name=s1_cfg['model_name'],
                          max_length=s1_cfg['max_seq_length'])
test_logits = collect_logits(s1_model, test_ds, DEVICE)
pred_cats_list = _apply_global_threshold(test_logits, threshold)

# Build gold pairs
gold_by_sent = {}
for r in test_sent:
    sent = r['sentence']
    if sent not in gold_by_sent:
        gold_by_sent[sent] = set()
    gold_by_sent[sent].add((r['category'], r['polarity']))

gold_cats_list = [set(cr['categories']) for cr in test_cat]
gold_pairs_list = [gold_by_sent.get(cr['sentence'], set()) for cr in test_cat]

cat_m = category_f1(pred_cats_list, gold_cats_list)
print(f'Stage 1 Cat F1: {cat_m["f1"]:.4f}')

# Free Stage 1 model
del s1_model, s1_ckpt
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# --- Build Stage 2 records ---

# Mode A: predicted categories (standard NB3)
stage2_pred_records = []
for cr, pred_cats in zip(test_cat, pred_cats_list):
    for cat in sorted(pred_cats):
        stage2_pred_records.append({
            'id': f"{cr['sentence_id']}_{cat}",
            'sentence': cr['sentence'],
            'category': cat,
            'polarity': 'positive',  # dummy
            'split': 'test',
        })

# Mode B: gold categories (oracle)
stage2_oracle_records = []
for r in test_sent:
    stage2_oracle_records.append({
        'id': r['id'],
        'sentence': r['sentence'],
        'category': r['category'],
        'polarity': r['polarity'],  # gold — used only for eval, not by model at inference
        'split': 'test',
    })

print(f'Stage 2 records — predicted: {len(stage2_pred_records)}, oracle: {len(stage2_oracle_records)}')

In [ ]:
# --- Load retrieval model and run all 3 conditions ---
retriever = Retriever(full_index, full_metadata,
                      top_k=ret_cfg['top_k'], threshold=ret_cfg['threshold'])

s2_ret_model = SentimentPredictor(
    model_name=s2_cfg['model_name'],
    num_sent_labels=s2_cfg['num_sent_labels'],
    embed_dim=s2_cfg.get('embed_dim', 64),
    tau=s2_cfg.get('tau', 0.05),
    dropout=s2_cfg.get('dropout', 0.1),
    use_retrieval=True,
    use_learnable_retriever=s2_cfg.get('use_learnable_retriever', False),
    margin=s2_cfg.get('rank_margin', 0.1),
    w_mode=s2_cfg.get('w_mode', 'full'),
    w_rank=s2_cfg.get('w_rank', 16),
).to(DEVICE)
s2_ret_state = torch.load('checkpoints/stage2_2014/best.pt', map_location=DEVICE)
s2_ret_model.load_state_dict(s2_ret_state, strict=False)
s2_ret_model.eval()
print('Retrieval model loaded')

# Condition 1: Retrieval + predicted categories
print('\nRunning: Retrieval + predicted categories...')
ret_pred_preds, ret_pred_probs = predict_sentiment_with_details(
    s2_ret_model, stage2_pred_records, retriever, embed_model,
    tokenizer_name=s2_cfg['model_name'],
    max_length=s2_cfg['max_seq_length'],
    top_k=ret_cfg['top_k'],
    device=DEVICE, use_retrieval=True,
    store_vectors=full_vectors)
print(f'  Predictions: {len(ret_pred_preds)}')

# Condition 2: Retrieval + oracle (gold) categories
print('Running: Retrieval + oracle categories...')
ret_oracle_preds, ret_oracle_probs = predict_sentiment_with_details(
    s2_ret_model, stage2_oracle_records, retriever, embed_model,
    tokenizer_name=s2_cfg['model_name'],
    max_length=s2_cfg['max_seq_length'],
    top_k=ret_cfg['top_k'],
    device=DEVICE, use_retrieval=True,
    store_vectors=full_vectors)
print(f'  Predictions: {len(ret_oracle_preds)}')

# Free retrieval model
del s2_ret_model, s2_ret_state
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Condition 3: No-retrieval + predicted categories
s2_noret_model = SentimentPredictor(
    model_name=s2_noret_cfg['model_name'],
    num_sent_labels=s2_noret_cfg['num_sent_labels'],
    embed_dim=s2_noret_cfg.get('embed_dim', 64),
    tau=s2_noret_cfg.get('tau', 0.05),
    dropout=s2_noret_cfg.get('dropout', 0.1),
    use_retrieval=False,
).to(DEVICE)
s2_noret_state = torch.load('checkpoints/stage2_2014_noret/best.pt', map_location=DEVICE)
s2_noret_model.load_state_dict(s2_noret_state, strict=False)
s2_noret_model.eval()
print('No-retrieval model loaded')

print('Running: No-retrieval + predicted categories...')
noret_pred_preds, noret_pred_probs = predict_sentiment_with_details(
    s2_noret_model, stage2_pred_records, None, None,
    tokenizer_name=s2_noret_cfg['model_name'],
    max_length=s2_noret_cfg['max_seq_length'],
    top_k=0,
    device=DEVICE, use_retrieval=False)
print(f'  Predictions: {len(noret_pred_preds)}')

del s2_noret_model, s2_noret_state
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# --- Compute metrics for all 3 conditions ---

def assemble_pairs_from_pred_cats(pred_cats_list, stage2_records, preds):
    """Assemble (cat, pol) pairs from Stage 1 predicted categories."""
    for rec, pred_idx in zip(stage2_records, preds):
        rec['predicted_polarity'] = ID2POL[pred_idx]
    pred_pairs_list = []
    rec_idx = 0
    for pred_cats in pred_cats_list:
        pairs = set()
        for cat in sorted(pred_cats):
            if rec_idx < len(stage2_records):
                pol = stage2_records[rec_idx].get('predicted_polarity', 'positive')
                pairs.add((cat, pol))
                rec_idx += 1
        pred_pairs_list.append(pairs)
    return pred_pairs_list


def compute_oracle_metrics(oracle_records, preds):
    """Compute Sent Acc on oracle (gold category) records."""
    correct = 0
    total = len(oracle_records)
    per_pol = defaultdict(lambda: {'correct': 0, 'total': 0})
    for rec, pred_idx in zip(oracle_records, preds):
        gold_pol = rec['polarity']
        pred_pol = ID2POL[pred_idx]
        per_pol[gold_pol]['total'] += 1
        if pred_pol == gold_pol:
            correct += 1
            per_pol[gold_pol]['correct'] += 1
    return {
        'accuracy': correct / total if total else 0,
        'correct': correct, 'total': total,
        'per_polarity': dict(per_pol),
    }


# Condition 1: Retrieval + predicted cats
ret_pred_pairs = assemble_pairs_from_pred_cats(
    pred_cats_list, stage2_pred_records, ret_pred_preds)
ret_pred_joint = joint_category_sentiment_f1(ret_pred_pairs, gold_pairs_list)
ret_pred_sent = sentiment_acc_given_correct_category(ret_pred_pairs, gold_pairs_list)

# Condition 2: Retrieval + oracle cats (direct accuracy, no Stage 1)
ret_oracle_metrics = compute_oracle_metrics(stage2_oracle_records, ret_oracle_preds)

# Condition 3: No-retrieval + predicted cats
noret_pred_pairs = assemble_pairs_from_pred_cats(
    pred_cats_list, stage2_pred_records, noret_pred_preds)
noret_pred_joint = joint_category_sentiment_f1(noret_pred_pairs, gold_pairs_list)
noret_pred_sent = sentiment_acc_given_correct_category(noret_pred_pairs, gold_pairs_list)

print('=' * 60)
print('EXP 1 SUMMARY: Oracle Category Analysis')
print('=' * 60)
print(f'  Retrieval  (pred cat):   Sent Acc|CC = {ret_pred_sent["accuracy"]:.4f} ({ret_pred_sent["correct"]}/{ret_pred_sent["total"]})')
print(f'  Retrieval  (oracle cat): Sent Acc    = {ret_oracle_metrics["accuracy"]:.4f} ({ret_oracle_metrics["correct"]}/{ret_oracle_metrics["total"]})')
print(f'  No-retrieval (pred cat): Sent Acc|CC = {noret_pred_sent["accuracy"]:.4f} ({noret_pred_sent["correct"]}/{noret_pred_sent["total"]})')
print()
gap_total = noret_pred_sent['accuracy'] - ret_pred_sent['accuracy']
gap_oracle = noret_pred_sent['accuracy'] - ret_oracle_metrics['accuracy']
gap_stage1 = ret_oracle_metrics['accuracy'] - ret_pred_sent['accuracy']
print(f'  Total gap (no-ret - ret):        {gap_total:+.4f} ({gap_total*100:+.1f}pp)')
print(f'  Gap after oracle (intrinsic):    {gap_oracle:+.4f} ({gap_oracle*100:+.1f}pp)')
print(f'  Stage 1 error contribution:      {gap_stage1:+.4f} ({gap_stage1*100:+.1f}pp)')
print()
print(f'  Joint F1 — Retrieval: {ret_pred_joint["f1"]:.4f}, No-ret: {noret_pred_joint["f1"]:.4f}')

In [ ]:
# --- Exp 4: Per-Polarity Breakdown ---

def per_polarity_from_pred_cats(pred_cats_list, gold_pairs_list, stage2_records, preds):
    """Per-polarity accuracy for predicted-category conditions."""
    per_pol = defaultdict(lambda: {'correct': 0, 'total': 0})
    confusion = defaultdict(int)  # (gold, pred) -> count
    rec_idx = 0
    for pred_cats, gold_pairs in zip(pred_cats_list, gold_pairs_list):
        gold_by_cat = {}
        for gc, gp in gold_pairs:
            gold_by_cat[gc] = gp
        for cat in sorted(pred_cats):
            if rec_idx < len(stage2_records):
                pred_pol = ID2POL[preds[rec_idx]]
                if cat in gold_by_cat:
                    gold_pol = gold_by_cat[cat]
                    per_pol[gold_pol]['total'] += 1
                    confusion[(gold_pol, pred_pol)] += 1
                    if pred_pol == gold_pol:
                        per_pol[gold_pol]['correct'] += 1
                rec_idx += 1
    return dict(per_pol), dict(confusion)


def print_confusion(confusion, label):
    pols = ['positive', 'negative', 'neutral']
    print(f'\n  Confusion matrix ({label}):')
    print(f'  {"":12s}  {"pred_pos":>8s} {"pred_neg":>8s} {"pred_neu":>8s}')
    for gp in pols:
        row = [confusion.get((gp, pp), 0) for pp in pols]
        total = sum(row)
        if total > 0:
            acc = row[pols.index(gp)] / total
            print(f'  gold_{gp[:3]:3s} ({total:3d})  {row[0]:>8d} {row[1]:>8d} {row[2]:>8d}  acc={acc:.3f}')


print('=' * 60)
print('EXP 4: Per-Polarity Breakdown')
print('=' * 60)

# Retrieval + predicted cats
ret_per_pol, ret_confusion = per_polarity_from_pred_cats(
    pred_cats_list, gold_pairs_list, stage2_pred_records, ret_pred_preds)
print('\n  Retrieval (predicted cats):')
for pol in ['positive', 'negative', 'neutral']:
    d = ret_per_pol.get(pol, {'correct': 0, 'total': 0})
    acc = d['correct'] / d['total'] if d['total'] else 0
    print(f'    {pol:10s}: {d["correct"]}/{d["total"]} = {acc:.4f}')
print_confusion(ret_confusion, 'Retrieval')

# No-retrieval + predicted cats
noret_per_pol, noret_confusion = per_polarity_from_pred_cats(
    pred_cats_list, gold_pairs_list, stage2_pred_records, noret_pred_preds)
print('\n  No-Retrieval (predicted cats):')
for pol in ['positive', 'negative', 'neutral']:
    d = noret_per_pol.get(pol, {'correct': 0, 'total': 0})
    acc = d['correct'] / d['total'] if d['total'] else 0
    print(f'    {pol:10s}: {d["correct"]}/{d["total"]} = {acc:.4f}')
print_confusion(noret_confusion, 'No-Retrieval')

# Oracle
print('\n  Retrieval (oracle cats):')
for pol in ['positive', 'negative', 'neutral']:
    d = ret_oracle_metrics['per_polarity'].get(pol, {'correct': 0, 'total': 0})
    acc = d['correct'] / d['total'] if d['total'] else 0
    print(f'    {pol:10s}: {d["correct"]}/{d["total"]} = {acc:.4f}')

---
## Exp 3: Ablate Text Signal vs Vector Signal

4 modes — no retraining, just patched inference.

In [ ]:
class SentimentDatasetDiag(SentimentDataset):
    """Diagnostic variant with independent control over text and vector signals."""
    
    def __init__(self, *args, use_text_signal=True, use_vector_signal=True, **kwargs):
        super().__init__(*args, **kwargs)
        self._use_text_signal = use_text_signal
        self._use_vector_signal = use_vector_signal
    
    def __getitem__(self, idx):
        record = self.records[idx]
        sentence = record['sentence']
        category = record['category']
        
        query_np = None
        neighbors = []
        if self.use_retrieval and self.top_k > 0:
            query_np, neighbors = self._retrieve_neighbors(sentence, category)
        
        cls_id = self.tokenizer.cls_token_id
        sep_id = self.tokenizer.sep_token_id
        
        sent_enc = self.tokenizer(sentence, add_special_tokens=False)
        cat_enc = self.tokenizer(category, add_special_tokens=False)
        
        token_ids = [cls_id] + sent_enc['input_ids'] + [sep_id] + \
                    cat_enc['input_ids'] + [sep_id]
        
        # TEXT SIGNAL: only append neighbor text if enabled
        if neighbors and self._use_text_signal:
            remaining = self.max_length - len(token_ids)
            per_nb = max(1, remaining // len(neighbors))
            for nb in neighbors:
                nb_enc = self.tokenizer(nb['sentence'], add_special_tokens=False)
                nb_ids = nb_enc['input_ids']
                if len(nb_ids) > per_nb - 1:
                    nb_ids = nb_ids[:per_nb - 1]
                token_ids.extend(nb_ids + [sep_id])
        
        if len(token_ids) > self.max_length:
            token_ids = token_ids[:self.max_length]
        
        pad_len = self.max_length - len(token_ids)
        attention_mask = [1] * len(token_ids) + [0] * pad_len
        token_ids = token_ids + [self.tokenizer.pad_token_id] * pad_len
        
        sentiment_label = POL2ID[record['polarity']]
        
        result = {
            'input_ids': torch.tensor(token_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'sentiment_label': torch.tensor(sentiment_label, dtype=torch.long),
        }
        
        if self.use_retrieval:
            # VECTOR SIGNAL: if disabled, pass zero vectors to trigger is_padding
            if self._use_vector_signal and neighbors:
                pol_ids = [POL2ID[nb['polarity']] for nb in neighbors]
                scores = [nb['score'] for nb in neighbors]
            else:
                pol_ids = []
                scores = []
            
            while len(pol_ids) < self.top_k:
                pol_ids.append(0)
                scores.append(float('-inf'))
            
            result['neighbor_polarities'] = torch.tensor(
                pol_ids[:self.top_k], dtype=torch.long)
            result['neighbor_scores'] = torch.tensor(
                scores[:self.top_k], dtype=torch.float32)
            result['query_polarity'] = torch.tensor(
                POL2ID[record['polarity']], dtype=torch.long)
            
            if self._use_vector_signal and query_np is not None:
                result['query_vec'] = torch.tensor(query_np[0], dtype=torch.float32)
            else:
                result['query_vec'] = torch.zeros(256, dtype=torch.float32)
            
            nb_vecs = []
            if self._use_vector_signal:
                for nb in neighbors[:self.top_k]:
                    fidx = nb.get('faiss_idx')
                    if (self.store_vectors is not None and fidx is not None
                            and fidx < len(self.store_vectors)):
                        nb_vecs.append(self.store_vectors[fidx].astype('float32'))
                    else:
                        nb_vecs.append(np.zeros(256, dtype='float32'))
            while len(nb_vecs) < self.top_k:
                nb_vecs.append(np.zeros(256, dtype='float32'))
            result['neighbor_vecs'] = torch.tensor(
                np.stack(nb_vecs[:self.top_k]), dtype=torch.float32)
        
        return result

print('SentimentDatasetDiag defined.')

In [ ]:
def run_ablation(model, records, retriever, embed_model, store_vectors,
                 cfg, ret_cfg, device, use_text, use_vector):
    """Run inference with ablated signals."""
    embed_device = 'cpu'
    if embed_model is not None:
        embed_device = next(embed_model.parameters()).device.type
    ds = SentimentDatasetDiag(
        records, retriever=retriever,
        tokenizer_name=cfg['model_name'],
        embedding_model=embed_model,
        store_vectors=store_vectors,
        max_length=cfg['max_seq_length'],
        top_k=ret_cfg.get('top_k', 5),
        device=embed_device,
        use_retrieval=True,
        use_text_signal=use_text,
        use_vector_signal=use_vector,
    )
    loader = DataLoader(ds, batch_size=16)
    all_preds = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch_gpu = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                         for k, v in batch.items()}
            out = model(
                input_ids=batch_gpu['input_ids'],
                attention_mask=batch_gpu['attention_mask'],
                neighbor_polarities=batch_gpu.get('neighbor_polarities'),
                neighbor_scores=batch_gpu.get('neighbor_scores'),
                query_vec=batch_gpu.get('query_vec'),
                neighbor_vecs=batch_gpu.get('neighbor_vecs'),
                query_polarity=batch_gpu.get('query_polarity'),
            )
            preds = out['logits'].argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
    return all_preds


# Reload retrieval model
s2_ret_model = SentimentPredictor(
    model_name=s2_cfg['model_name'],
    num_sent_labels=s2_cfg['num_sent_labels'],
    embed_dim=s2_cfg.get('embed_dim', 64),
    tau=s2_cfg.get('tau', 0.05),
    dropout=s2_cfg.get('dropout', 0.1),
    use_retrieval=True,
    use_learnable_retriever=s2_cfg.get('use_learnable_retriever', False),
    margin=s2_cfg.get('rank_margin', 0.1),
    w_mode=s2_cfg.get('w_mode', 'full'),
    w_rank=s2_cfg.get('w_rank', 16),
).to(DEVICE)
s2_ret_state = torch.load('checkpoints/stage2_2014/best.pt', map_location=DEVICE)
s2_ret_model.load_state_dict(s2_ret_state, strict=False)
s2_ret_model.eval()
print('Retrieval model reloaded for ablation')

# Use oracle records for cleaner ablation (remove Stage 1 noise)
ablation_records = stage2_oracle_records

modes = [
    ('text+vector (standard)', True, True),
    ('text-only',              True, False),
    ('vector-only',            False, True),
    ('neither',                False, False),
]

ablation_results = {}
for name, use_text, use_vector in modes:
    print(f'\nRunning ablation: {name}...')
    preds = run_ablation(
        s2_ret_model, ablation_records, retriever, embed_model, full_vectors,
        s2_cfg, ret_cfg, DEVICE, use_text, use_vector)
    metrics = compute_oracle_metrics(ablation_records, preds)
    ablation_results[name] = metrics
    print(f'  Acc: {metrics["accuracy"]:.4f} ({metrics["correct"]}/{metrics["total"]})')

del s2_ret_model, s2_ret_state
gc.collect()
torch.cuda.empty_cache()

In [ ]:
print('=' * 60)
print('EXP 3 SUMMARY: Signal Ablation (oracle categories)')
print('=' * 60)
print(f'{"Mode":30s} {"Acc":>8s} {"Correct":>8s} {"Total":>6s}')
print('-' * 60)
for name, m in ablation_results.items():
    print(f'{name:30s} {m["accuracy"]:>8.4f} {m["correct"]:>8d} {m["total"]:>6d}')
print()

# Per-polarity for each mode
for name, m in ablation_results.items():
    print(f'\n  {name}:')
    for pol in ['positive', 'negative', 'neutral']:
        d = m['per_polarity'].get(pol, {'correct': 0, 'total': 0})
        acc = d['correct'] / d['total'] if d['total'] else 0
        print(f'    {pol:10s}: {d["correct"]}/{d["total"]} = {acc:.4f}')

---
## Exp 5 + 6: Error Analysis + Confidence

In [ ]:
# --- Exp 5: Hurt cases (no-ret correct, retrieval wrong) ---
# Use predicted-category records since that's the real deployment scenario

# Build per-record gold lookup from pred_cats condition
gold_by_cat_per_sent = {}
for r in test_sent:
    key = r['sentence']
    if key not in gold_by_cat_per_sent:
        gold_by_cat_per_sent[key] = {}
    gold_by_cat_per_sent[key][r['category']] = r['polarity']

hurt_cases = []
help_cases = []
for i, rec in enumerate(stage2_pred_records):
    sent = rec['sentence']
    cat = rec['category']
    
    gold_pol = gold_by_cat_per_sent.get(sent, {}).get(cat)
    if gold_pol is None:
        continue  # category was incorrectly predicted by Stage 1
    
    ret_pol = ID2POL[ret_pred_preds[i]]
    noret_pol = ID2POL[noret_pred_preds[i]]
    
    if noret_pol == gold_pol and ret_pol != gold_pol:
        hurt_cases.append({
            'idx': i, 'sentence': sent, 'category': cat,
            'gold_polarity': gold_pol,
            'ret_pred': ret_pol, 'noret_pred': noret_pol,
        })
    elif ret_pol == gold_pol and noret_pol != gold_pol:
        help_cases.append({
            'idx': i, 'sentence': sent, 'category': cat,
            'gold_polarity': gold_pol,
            'ret_pred': ret_pol, 'noret_pred': noret_pol,
        })

print(f'Records where no-ret correct, ret wrong (HURT): {len(hurt_cases)}')
print(f'Records where ret correct, no-ret wrong (HELP): {len(help_cases)}')
print(f'Net effect: {len(help_cases) - len(hurt_cases):+d} records')

In [ ]:
# Retrieve neighbors for hurt cases to understand why retrieval fails
print(f'\nAnalyzing {len(hurt_cases)} hurt cases...')

hurt_error_directions = Counter()
hurt_pol_matches = []

for case in hurt_cases:
    sent = case['sentence']
    cat = case['category']
    
    tok_enc = tokenizer(
        sent, cat,
        max_length=128, padding=False, truncation=True,
        return_tensors='pt')
    with torch.no_grad():
        query_vec = embed_model.encode(
            tok_enc['input_ids'].to(DEVICE),
            tok_enc['attention_mask'].to(DEVICE))
    query_np = query_vec.cpu().numpy().astype('float32')
    neighbors = retriever.retrieve(query_np, exclude_sentence=sent)
    
    case['neighbors'] = [{
        'sentence': nb['sentence'][:80],
        'category': nb.get('aspect_category', nb.get('category')),
        'polarity': nb['polarity'],
        'score': round(nb['score'], 4)
    } for nb in neighbors]
    
    if neighbors:
        pm5 = sum(1 for nb in neighbors if nb['polarity'] == case['gold_polarity']) / len(neighbors)
    else:
        pm5 = 0.0
    case['pol_match_5'] = pm5
    hurt_pol_matches.append(pm5)
    hurt_error_directions[(case['gold_polarity'], case['ret_pred'])] += 1

print(f'\nHurt cases — mean pol_match@5: {np.mean(hurt_pol_matches):.4f}')
print(f'Hurt cases — pol_match@5 < 0.5: {sum(1 for p in hurt_pol_matches if p < 0.5)}/{len(hurt_pol_matches)}')
print(f'\nError directions (gold → ret_pred):')
for (g, p), cnt in hurt_error_directions.most_common():
    print(f'  {g:10s} → {p:10s}: {cnt}')

In [ ]:
# Print first 10 hurt cases with neighbors
print('\n=== SAMPLE HURT CASES (first 10) ===')
for i, case in enumerate(hurt_cases[:10]):
    print(f'\n--- Case {i+1} ---')
    print(f'  Sentence: {case["sentence"][:100]}')
    print(f'  Category: {case["category"]}')
    print(f'  Gold: {case["gold_polarity"]}  |  Ret pred: {case["ret_pred"]}  |  No-ret pred: {case["noret_pred"]}')
    print(f'  pol_match@5: {case["pol_match_5"]:.2f}')
    for j, nb in enumerate(case.get('neighbors', [])):
        print(f'    nb{j+1}: [{nb["polarity"]:8s}] (sim={nb["score"]:.4f}) {nb["sentence"]}')

In [ ]:
# --- Exp 6: Confidence Analysis ---

# Max softmax probability for retrieval vs no-retrieval
ret_max_probs = ret_pred_probs.max(dim=-1).values.numpy()
noret_max_probs = noret_pred_probs.max(dim=-1).values.numpy()

# Build correctness masks (for records with gold category match)
ret_correct_mask = []
noret_correct_mask = []
for i, rec in enumerate(stage2_pred_records):
    gold_pol = gold_by_cat_per_sent.get(rec['sentence'], {}).get(rec['category'])
    if gold_pol is None:
        ret_correct_mask.append(None)
        noret_correct_mask.append(None)
    else:
        ret_correct_mask.append(ID2POL[ret_pred_preds[i]] == gold_pol)
        noret_correct_mask.append(ID2POL[noret_pred_preds[i]] == gold_pol)

# Filter to only records with gold category
ret_conf_correct = [ret_max_probs[i] for i, m in enumerate(ret_correct_mask) if m is True]
ret_conf_wrong = [ret_max_probs[i] for i, m in enumerate(ret_correct_mask) if m is False]
noret_conf_correct = [noret_max_probs[i] for i, m in enumerate(noret_correct_mask) if m is True]
noret_conf_wrong = [noret_max_probs[i] for i, m in enumerate(noret_correct_mask) if m is False]

print('=' * 60)
print('EXP 6: Confidence Analysis')
print('=' * 60)
print(f'  Retrieval:')
print(f'    Overall mean confidence:      {np.mean(ret_max_probs):.4f}')
print(f'    Correct predictions (n={len(ret_conf_correct):3d}): {np.mean(ret_conf_correct):.4f}')
print(f'    Wrong predictions   (n={len(ret_conf_wrong):3d}): {np.mean(ret_conf_wrong):.4f}')
print(f'  No-Retrieval:')
print(f'    Overall mean confidence:      {np.mean(noret_max_probs):.4f}')
print(f'    Correct predictions (n={len(noret_conf_correct):3d}): {np.mean(noret_conf_correct):.4f}')
print(f'    Wrong predictions   (n={len(noret_conf_wrong):3d}): {np.mean(noret_conf_wrong):.4f}')

# Confidence on hurt cases specifically
hurt_ret_confs = [ret_max_probs[c['idx']] for c in hurt_cases]
hurt_noret_confs = [noret_max_probs[c['idx']] for c in hurt_cases]
print(f'\n  Hurt cases ({len(hurt_cases)} records):')
print(f'    Ret confidence (wrong):   {np.mean(hurt_ret_confs):.4f}')
print(f'    No-ret confidence (right): {np.mean(hurt_noret_confs):.4f}')

---
## Final Summary Report

In [ ]:
print()
print('=' * 70)
print('         DIAGNOSTIC REPORT: Why Retrieval Loses by 9.5pp')
print('=' * 70)

# Exp 2
train_pm5 = np.mean([r['pol_match_5'] for r in train_quality])
test_pm5 = np.mean([r['pol_match_5'] for r in test_quality])
test_sim = np.mean([r['avg_sim'] for r in test_quality])
n_few = sum(1 for r in test_quality if r['n_valid'] < 3)
print(f'\n[Exp 2] RETRIEVAL QUALITY')
print(f'  Train pol_match@5: {train_pm5:.4f}')
print(f'  Test  pol_match@5: {test_pm5:.4f}  (gap: {test_pm5-train_pm5:+.4f})')
print(f'  Test  avg cosine:  {test_sim:.4f}')
print(f'  Test records with <3 valid neighbors: {n_few}/{len(test_quality)}')
for pol in ['positive', 'negative', 'neutral']:
    subset = [r for r in test_quality if r['polarity'] == pol]
    if subset:
        pm5 = np.mean([r['pol_match_5'] for r in subset])
        print(f'    {pol:10s} (n={len(subset):3d}): pol_match@5={pm5:.4f}')

# Exp 1
print(f'\n[Exp 1] STAGE 1 ERROR IMPACT')
print(f'  Retrieval  (pred cat):   {ret_pred_sent["accuracy"]:.4f}')
print(f'  Retrieval  (oracle cat): {ret_oracle_metrics["accuracy"]:.4f}')
print(f'  No-retrieval (pred cat): {noret_pred_sent["accuracy"]:.4f}')
print(f'  -> Stage 1 error:     {gap_stage1*100:+.1f}pp')
print(f'  -> Intrinsic gap:     {gap_oracle*100:+.1f}pp')

# Exp 3
print(f'\n[Exp 3] SIGNAL ABLATION (oracle categories)')
for name, m in ablation_results.items():
    print(f'  {name:30s}: {m["accuracy"]:.4f}')

# Exp 5
print(f'\n[Exp 5] ERROR ANALYSIS')
print(f'  Hurt cases (no-ret right, ret wrong): {len(hurt_cases)}')
print(f'  Help cases (ret right, no-ret wrong):  {len(help_cases)}')
print(f'  Net: {len(help_cases)-len(hurt_cases):+d}')
print(f'  Mean pol_match@5 in hurt cases: {np.mean(hurt_pol_matches):.4f}')
print(f'  Hurt cases with pol_match@5 < 0.5: {sum(1 for p in hurt_pol_matches if p < 0.5)}/{len(hurt_pol_matches)}')
print(f'  Top error directions:')
for (g, p), cnt in hurt_error_directions.most_common(3):
    print(f'    {g} -> {p}: {cnt}')

# Exp 6
print(f'\n[Exp 6] CONFIDENCE')
print(f'  Mean max_prob — Ret: {np.mean(ret_max_probs):.4f}, No-ret: {np.mean(noret_max_probs):.4f}')
print(f'  Wrong preds  — Ret: {np.mean(ret_conf_wrong):.4f}, No-ret: {np.mean(noret_conf_wrong):.4f}')

print(f'\n{"=" * 70}')
print('ROOT CAUSE DETERMINATION')
print(f'{"=" * 70}')
print('Based on the experiments above, the primary factor(s) are:')
print(f'  1. Retrieval quality gap (train vs test pol_match@5): {test_pm5-train_pm5:+.4f}')
print(f'  2. Stage 1 error contribution: {gap_stage1*100:+.1f}pp')
print(f'  3. Intrinsic architecture gap (oracle): {gap_oracle*100:+.1f}pp')
std_acc = ablation_results.get('text+vector (standard)', {}).get('accuracy', 0)
to_acc = ablation_results.get('text-only', {}).get('accuracy', 0)
vo_acc = ablation_results.get('vector-only', {}).get('accuracy', 0)
print(f'  4. Text signal delta (text-only vs standard): {to_acc - std_acc:+.4f}')
print(f'  5. Vector signal delta (vector-only vs standard): {vo_acc - std_acc:+.4f}')
print()
print('See individual experiment sections above for detailed breakdowns.')

---
## Exp 7: CLS 768-dim from No-Ret DeBERTa as Retrieval Vector

**Hypothesis:** The no-ret model's DeBERTa (finetuned with classification loss) encodes polarity in CLS 768-dim that generalizes to unseen test sentences — unlike the contrastive projection head (256-dim) which memorizes train positions.

**Method:** Extract CLS vectors from no-ret DeBERTa (no projection head), build FAISS IndexFlatIP (768-dim), measure pol_match@5 on test. Compare with contrastive embedding 256-dim baseline.

In [ ]:
import faiss as _faiss
from torch.utils.data import DataLoader, Dataset as _Dataset

# 1. Load no-ret DeBERTa encoder
_noret_model = SentimentPredictor(
    model_name=s2_noret_cfg['model_name'],
    num_sent_labels=s2_noret_cfg['num_sent_labels'],
    use_retrieval=False,
).to(DEVICE)
_noret_state = torch.load('checkpoints/stage2_2014_noret/best.pt', map_location=DEVICE)
_noret_model.load_state_dict(_noret_state, strict=False)
_noret_model.eval()
_encoder = _noret_model.encoder
print(f'No-ret DeBERTa loaded, hidden_size={_encoder.config.hidden_size}')

# 2. Encode function: CLS 768-dim, L2 normalized
class _RecDS(_Dataset):
    def __init__(self, recs, tok, ml=128):
        self.recs = recs
        self.tok = tok
        self.ml = ml
    def __len__(self):
        return len(self.recs)
    def __getitem__(self, idx):
        r = self.recs[idx]
        cat = r.get('category', r.get('aspect_category', ''))
        enc = self.tok(r['sentence'], cat,
                       max_length=self.ml, padding='max_length',
                       truncation=True, return_tensors='pt')
        return {k: v.squeeze(0) for k, v in enc.items()}

def encode_cls_768(records, encoder, tok, max_length=128, batch_size=32, device='cuda'):
    ds = _RecDS(records, tok, max_length)
    loader = DataLoader(ds, batch_size=batch_size)
    all_vecs = []
    encoder.eval()
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            out = encoder(input_ids=ids, attention_mask=mask)
            cls = out.last_hidden_state[:, 0]
            cls = torch.nn.functional.normalize(cls, p=2, dim=-1)
            all_vecs.append(cls.cpu().numpy())
    return np.concatenate(all_vecs, axis=0).astype('float32')

# 3. Encode train (all 3516) and test (973)
print('Encoding train records (CLS 768-dim)...')
train_cls_vecs = encode_cls_768(train_sent, _encoder, tokenizer, device=DEVICE)
print(f'  Train: {train_cls_vecs.shape}')

print('Encoding test records (CLS 768-dim)...')
test_cls_vecs = encode_cls_768(test_sent, _encoder, tokenizer, device=DEVICE)
print(f'  Test: {test_cls_vecs.shape}')

del _noret_model, _noret_state, _encoder
gc.collect()
torch.cuda.empty_cache()
print('Model freed.')

# 4. Build FAISS 768-dim from train
_faiss.normalize_L2(train_cls_vecs)
cls_index = _faiss.IndexFlatIP(train_cls_vecs.shape[1])
cls_index.add(train_cls_vecs)
cls_meta = [{'polarity': r['polarity'],
             'category': r.get('category', r.get('aspect_category')),
             'sentence': r['sentence']}
            for r in train_sent]
print(f'FAISS index: {cls_index.ntotal} vectors, dim={cls_index.d}')

# 5. Test → train search
K = 5
_faiss.normalize_L2(test_cls_vecs)
D_test, I_test = cls_index.search(test_cls_vecs, K)

# 6. pol_match@5 per polarity (test)
test_pm = defaultdict(list)
for i, rec in enumerate(test_sent):
    q_pol = rec['polarity']
    matches = sum(1 for j in I_test[i] if j >= 0 and cls_meta[j]['polarity'] == q_pol)
    test_pm[q_pol].append(matches / K)

# cat_match@5 (test)
test_cm = defaultdict(list)
for i, rec in enumerate(test_sent):
    q_cat = rec.get('category', rec.get('aspect_category'))
    matches = sum(1 for j in I_test[i] if j >= 0 and cls_meta[j]['category'] == q_cat)
    test_cm[q_cat].append(matches / K)

# 7. Train self-eval (with self-exclusion)
D_train, I_train = cls_index.search(train_cls_vecs, K + 1)
train_pm = defaultdict(list)
for i, rec in enumerate(train_sent):
    q_pol = rec['polarity']
    neighbors = [j for j in I_train[i] if j != i][:K]
    matches = sum(1 for j in neighbors if j >= 0 and cls_meta[j]['polarity'] == q_pol)
    train_pm[q_pol].append(matches / K)

# 8. Cosine distributions
test_scores = D_test[D_test > -1e9]
rng = np.random.default_rng(42)
rand_idx = rng.integers(0, len(train_cls_vecs), size=(500, 2))
rand_sims = np.array([float(train_cls_vecs[a] @ train_cls_vecs[b])
                       for a, b in rand_idx if a != b])

# === RESULTS ===
contr_baseline = {'positive': 0.897, 'negative': 0.580, 'neutral': 0.383, 'overall': 0.775}

print(f'\n{"="*70}')
print(f'EXP 7: CLS 768-dim (No-Ret DeBERTa) vs Contrastive 256-dim')
print(f'{"="*70}')

print(f'\nTEST pol_match@{K}:')
print(f'{"Polarity":<12} {"n":<6} {"CLS 768":<10} {"Contr 256":<10} {"Delta":<10}')
print(f'{"-"*50}')
overall_cls = np.mean([m for ms in test_pm.values() for m in ms])
for pol in ['positive', 'negative', 'neutral']:
    if pol in test_pm:
        val = np.mean(test_pm[pol])
        base = contr_baseline[pol]
        delta = val - base
        print(f'{pol:<12} {len(test_pm[pol]):<6} {val:<10.3f} {base:<10.3f} {delta:+.3f}')
print(f'{"-"*50}')
d = overall_cls - contr_baseline['overall']
print(f'{"Overall":<12} {len(test_sent):<6} {overall_cls:<10.3f} {contr_baseline["overall"]:<10.3f} {d:+.3f}')

print(f'\nTEST cat_match@{K}:')
overall_cm = np.mean([m for ms in test_cm.values() for m in ms])
for cat in sorted(test_cm.keys()):
    val = np.mean(test_cm[cat])
    print(f'  {cat:<30s} (n={len(test_cm[cat]):3d}): {val:.3f}')
print(f'  Overall: {overall_cm:.3f}')

print(f'\nTRAIN pol_match@{K} (self-excluded):')
train_overall = np.mean([m for ms in train_pm.values() for m in ms])
for pol in ['positive', 'negative', 'neutral']:
    if pol in train_pm:
        val = np.mean(train_pm[pol])
        print(f'  {pol:<12} (n={len(train_pm[pol])}): {val:.3f}')
print(f'  Overall: {train_overall:.3f}')
print(f'  Train-Test gap: {train_overall - overall_cls:+.3f}')

print(f'\nCosine distribution:')
print(f'  Test→train:  mean={test_scores.mean():.4f}, std={test_scores.std():.4f} (contr: 0.9937)')
print(f'  Random train: mean={rand_sims.mean():.4f}, std={rand_sims.std():.4f} (contr polonly: 0.4242)')

del cls_index, train_cls_vecs, test_cls_vecs
gc.collect()